# Kiên Đoàn TTS — Unified (7 giọcng)

> **GPU:** T4 16GB (tự động) · **Thời gian:** ~1 phút lân đầu, ~5s/lân sau

## Hướng dẫn

1. **Runtime** → **Run all** (Ctrl+F9)
2. Chọn giọcng → Nghe thử
3. Nhập văn bản → Tạo giọcng nói

> Powered by OmniVoice (MIT License) — github.com/k2-fsa/OmniVoice

In [ ]:
print('Cài đặt (~1 phút)...')
!pip install -q omnivoice gradio "numpy<2.1" "requests==2.32.4"
print('Cài đặt hoàn tất!')

# Tải 7 giọcng mẫu
VOICES = ['nam-cong-nghe', 'minh-anh', 'thanh-nien-tu-tin',
          'nam-tram-am', 'adam', 'ngoc-huyen', 'nho-ngot-ngao']
BASE = 'https://raw.githubusercontent.com/doanquangkien/voice-notebooks/main/samples'

import os
os.makedirs('samples', exist_ok=True)

for v in VOICES:
    path = f'samples/{v}.mp3'
    if not os.path.exists(path):
        !wget -q {BASE}/{v}.mp3 -O {path}
        print(f'  ✓ {v}')
    else:
        print(f'  • {v} (có sẵn)')

print(f'Đã tải {len(VOICES)} giọcng mẫu!')

In [ ]:
print('Đãng khởi động OmniVoice... (~30 giây)')

import logging, json, time, re
import numpy as np
import torch

# Patch: torch._utils removed in torch 2.13+
import torch as _torch
if not hasattr(_torch, '_utils'):
    _torch._utils = _torch._C._utils

# Shim: AutoFeatureExtractor removed in transformers 5.x
import transformers as _tf
class _SafeAutoFeatureExtractor:
    @staticmethod
    def from_pretrained(model_name, **kwargs):
        try:
            from transformers import AutoConfig
            cfg = AutoConfig.from_pretrained(model_name, trust_remote_code=True, **kwargs)
            sr = getattr(cfg, 'sampling_rate', 24000)
        except Exception:
            sr = 24000
        class _Result:
            sampling_rate = sr
        return _Result()
_tf.AutoFeatureExtractor = _SafeAutoFeatureExtractor

from omnivoice import OmniVoice, OmniVoiceGenerationConfig
from omnivoice.utils.common import get_best_device

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

# Wait for GPU
for i in range(30):
    if torch.cuda.is_available():
        print(f'GPU: {torch.cuda.get_device_name(0)}')
        break
    time.sleep(1)
else:
    raise RuntimeError('Cần GPU! Runtime > Change runtime type > T4 GPU')

# Load model 1 lần
DEVICE = get_best_device()
logger.info(f'Loading OmniVoice on {DEVICE}...')
model = OmniVoice.from_pretrained(
    'k2-fsa/OmniVoice', device_map=DEVICE, dtype=torch.float16, load_asr=True
)
SR = model.sampling_rate
logger.info(f'Model ready — SR: {SR}Hz')

# Load voice config tu GitHub
CONFIG_URL = 'https://raw.githubusercontent.com/doanquangkien/voice-notebooks/main/voice_config.json'
DEFAULT_CFG = dict(num_step=32, guidance_scale=2.0, speed=1.0,
                   denoise=True, preprocess_prompt=True, postprocess_output=True,
                   position_temperature=5.0, class_temperature=0.2,
                   pad_duration=0.1, fade_duration=0.1)
try:
    import urllib.request
    cfg_data = json.loads(urllib.request.urlopen(CONFIG_URL).read())
    VOICE_CFG = {**DEFAULT_CFG, **cfg_data}
    print(f'✓ Config từ GitHub: guidance={VOICE_CFG["guidance_scale"]}, speed={VOICE_CFG["speed"]}, steps={VOICE_CFG["num_step"]}')
except Exception:
    VOICE_CFG = DEFAULT_CFG
    print(f'• Config default: guidance={VOICE_CFG["guidance_scale"]}, speed={VOICE_CFG["speed"]}')

GEN_CFG = OmniVoiceGenerationConfig(
    num_step=VOICE_CFG['num_step'],
    guidance_scale=VOICE_CFG['guidance_scale'],
    denoise=VOICE_CFG['denoise'],
    preprocess_prompt=VOICE_CFG['preprocess_prompt'],
    postprocess_output=VOICE_CFG['postprocess_output'],
    position_temperature=VOICE_CFG['position_temperature'],
    class_temperature=VOICE_CFG['class_temperature'],
    pad_duration=VOICE_CFG['pad_duration'],
    fade_duration=VOICE_CFG['fade_duration'],
)
SPEED = VOICE_CFG['speed']

# Tạo 7 VoiceClonePrompts
print('\nTạo voice prompts...')
VOICE_PROMPTS = {}
for v in VOICES:
    VOICE_PROMPTS[v] = model.create_voice_clone_prompt(ref_audio=f'samples/{v}.mp3')
    print(f'  ✓ {v}')

# Định nghĩa 7 giọcng
VOICE_NAMES = {
    'nam-cong-nghe':      'Nam công nghệ',
    'minh-anh':           'Minh Anh',
    'thanh-nien-tu-tin':  'Thanh niên túng tin',
    'nam-tram-am':        'Nam trầm ăm',
    'adam':               'Adam',
    'ngoc-huyen':         'Ngọc Huyền',
    'nho-ngot-ngao':      'Nhỏ Ngọt Ngào',
}

# IPython UI
from ipywidgets import widgets
from IPython.display import display, Audio, HTML

header_html = HTML('''<div style="background:linear-gradient(135deg,#6366f1,#8b5cf6);
    border-radius:12px;padding:16px 20px;margin-bottom:16px;color:white;
    font-family:system-ui,-apple-system,sans-serif">
    <div style="font-size:20px;font-weight:700">🎙️ Kiên Đoàn TTS — Unified</div>
    <div style="font-size:13px;opacity:0.9;margin-top:4px">
        {count} giọcng · OmniVoice 0.6B · Config từ GitHub</div>
</div>'''.format(count=len(VOICES)))

voice_label = widgets.HTML('<div style="font-weight:600;margin:8px 0 4px">🎤 Chọn giọcng:</div>')
voice_radio = widgets.RadioButtons(
    options=[(VOICE_NAMES[v], v) for v in VOICES],
    layout=widgets.Layout(width='100%')
)

preview_btn = widgets.Button(description='🔊 Nghe thử',
    button_style='info',
    layout=widgets.Layout(width='120px', height='34px'))
sample_out = widgets.Output()

text_label = widgets.HTML('<div style="font-weight:600;margin:12px 0 4px">📝 Nhập văn bản:</div>')
text_input = widgets.Textarea(
    placeholder='Nhập văn bản muốn chuyển thành giọcng nói...',
    layout=widgets.Layout(width='100%', height='100px')
)

generate_btn = widgets.Button(description='▶ Tạo giọcng nói',
    button_style='primary',
    layout=widgets.Layout(width='160px', height='38px'))
audio_out = widgets.Output()

status_html = widgets.HTML('')

ui = widgets.VBox([
    header_html, voice_label, voice_radio,
    widgets.HBox([preview_btn]), sample_out,
    text_label, text_input,
    widgets.HBox([generate_btn]),
    status_html, audio_out
], layout=widgets.Layout(max_width='600px', padding='8px'))

display(ui)

In [ ]:
def _on_preview(b):
    with sample_out:
        sample_out.clear_output()
        key = voice_radio.value
        display(Audio(f'samples/{key}.mp3', autoplay=True))

def _on_generate(b):
    with audio_out:
        audio_out.clear_output()
        status_html.value = ''
        key = voice_radio.value
        text = text_input.value.strip()
        if not text:
            status_html.value = '<div style="color:#e74c3c;font-weight:600">⚠ Vui lòng nhập văn bản</div>'
            return
        name = VOICE_NAMES[key]
        status_html.value = f'<div style="color:#6366f1">⏳ Đang tạo {name}...</div>'
        try:
            start = time.time()
            paragraphs = [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]
            prompt = VOICE_PROMPTS[key]
            if len(paragraphs) == 1:
                audio = model.generate(
                    text=paragraphs[0], voice_clone_prompt=prompt,
                    language='vi', speed=SPEED, generation_config=GEN_CFG
                )[0]
            else:
                audios = []
                for i, p in enumerate(paragraphs):
                    a = model.generate(
                        text=p, voice_clone_prompt=prompt,
                        language='vi', speed=SPEED, generation_config=GEN_CFG
                    )[0]
                    audios.append(a)
                    if i < len(paragraphs) - 1:
                        audios.append(np.zeros(int(SR * 0.3)))
                audio = np.concatenate(audios)
            elapsed = time.time() - start
            duration = len(audio) / SR
            waveform = (audio * 32767).astype(np.int16)
            status_html.value = (
                f'<div style="color:#27ae60;font-weight:600">✓ {name} — '
                f'{duration:.1f}s audio, {elapsed:.1f}s tạo</div>'
            )
            display(Audio(data=waveform, rate=SR, autoplay=True))
        except Exception as e:
            status_html.value = f'<div style="color:#e74c3c">✗ Lỗi: {e}</div>'

preview_btn.on_click(_on_preview)
generate_btn.on_click(_on_generate)
print('Sẵn sàng! Chọn giọcng → Nhập văn bản → Tạo giọcng nói')